In [38]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Set rendering default to browser view
pio.renderers.default = "colab"

# 1. Load the stored data packet vectors
# Replace filename with your latest lockin saved output file name
da = np.load("pll_final_version_vanity.npz")

data      = da["data_main"].astype(np.uint64)
data_freq = da["data_freq"].astype(np.uint64)
data_mix    = d["data_mix"].astype(np.uint16)
data_inp    = d["data_inp"].astype(np.uint16)
fs        = d["fs"]

# Compute continuous time arrays
t = np.arange(len(data)) / fs
t_freq = np.arange(len(data_freq)) / fs

ValueError: Cannot load file containing pickled data when allow_pickle=False

In [20]:
# ============================================================
# ===== FIXED DECODING - NEW LOCK-IN STREAM LAYOUT =====
# ============================================================
# Define your hardware's peak input voltage range
# (Set to 1.0 if you just want normalized -1 to +1 range, or 1.0 for Red Pitaya RF inputs)
V_PEAK = 1.0
INT16_FS = 32767.0

# --- MAIN STREAM UNPACKING [data_main] ---
# 1. Extract raw 16-bit signed integer components first
q_raw   = ((data >> 48) & 0xFFFF).astype(np.int16)
i_raw   = ((data >> 32) & 0xFFFF).astype(np.int16)
adc_raw = ((data >> 16) & 0xFFFF).astype(np.int16)
sin_raw = (data & 0xFFFF).astype(np.int16)

data_inp = (data_inp & 0xFFFF).astype(np.int16)
data_m = (data_mix & 0xFFFF).astype(np.int16)

# 2. Convert to physical Volts (cast to float32 during math operation)
mix_q_lpf   = (q_raw / INT16_FS) * V_PEAK
mix_i_lpf   = (i_raw / INT16_FS) * V_PEAK
adc_aligned = (adc_raw / INT16_FS) * V_PEAK
sin_ref     = (sin_raw / INT16_FS) * V_PEAK

data_i = (data_inp / INT16_FS) * V_PEAK
data_m = (data_m / INT16_FS) * V_PEAK

calculated_amplitude = 2.0 * np.sqrt(mix_i_lpf.astype(np.float32)**2 + mix_q_lpf.astype(np.float32)**2)
'''
i_rr =[]

for i in data_i:
  i_0   = ((i >> 48) & 0xFFFF).astype(np.int16)
  i_1   = ((i >> 32) & 0xFFFF).astype(np.int16)
  i_2 = ((i >> 16) & 0xFFFF).astype(np.int16)
  i_3 = (i & 0xFFFF).astype(np.int16)
  i_rr.append(i_3)


q_rr =[]
for q in data_q:
  q_0   = ((q >> 48) & 0xFFFF).astype(np.int16)
  q_1   = ((q >> 32) & 0xFFFF).astype(np.int16)
  q_2 = ((q >> 16) & 0xFFFF).astype(np.int16)
  q_3 = (q & 0xFFFF).astype(np.int16)
  q_rr.append(q_3)



mix_q   = (np.array(q_rr) / INT16_FS) * V_PEAK
mix_i   = (np.array(i_rr) / INT16_FS) * V_PEAK

l = len(mix_q)


# Calculate Vector Amplitude (R) from the filtered I and Q quadratures
# Multiplied by 2 to account for the mixing loss factor inherent to analog multiplication

calculated_amplitude_raw = 2.0 * np.sqrt(mix_i.astype(np.float32)**2 + mix_q.astype(np.float32)**2)
'''
# --- FREQUENCY STREAM UNPACKING [data_freq] ---
period_avg   = ((data_freq >> 48) & 0xFFFF).astype(np.uint16)
ref_edge_pll = (data_freq >> 47) & 0x1
nco_edge_pll = (data_freq >> 46) & 0x1
ftw_selected = data_freq & ((1 << 46) - 1)

# Convert 46-bit FTW back to physical Hz metrics
FCW = 46
freq_from_ftw = (ftw_selected * fs) / (2**FCW)

In [ ]:
print(i_rr[0:10])
print(q_rr[0:10])

[np.int16(0), np.int16(0), np.int16(0), np.int16(1857), np.int16(0), np.int16(0), np.int16(0), np.int16(1916), np.int16(0), np.int16(0)]
[np.int16(0), np.int16(0), np.int16(0), np.int16(1600), np.int16(0), np.int16(0), np.int16(0), np.int16(1605), np.int16(0), np.int16(0)]


In [21]:
# ============================================================
# ===== PLOTLY DASHBOARD GENERATION =====
# ============================================================

fig = make_subplots(
    rows=8, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    subplot_titles=(
        "1. Real-time Extracted Target Amplitude (Calculated R Vector)",
        "2. Low-Pass Filtered Quadrature Components (DC Levels)",
        "3. Raw Phase Checkpoint Alignment (ADC vs Reference Sine)",
        "4. Loop Pulse Engine Edge Checkpoints (ADPLL Control Channels)",
        "5. Running Coarse Frame Estimator Timing Windows (Period)",
        "6. Final Resolved System Clock Frequency (Tracking Output)"
    )
)

# Row 1: Vector Amplitude Metrics
fig.add_trace(go.Scatter(x=t, y=calculated_amplitude, name="Calculated Amplitude (R)", line=dict(color="darkgreen", width=2.5)), row=1, col=1)

# Row 2: Quadrature components (I & Q)
fig.add_trace(go.Scatter(x=t, y=mix_i_lpf, name="In-Phase Filtered Line (I)", line=dict(color="blue")), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=mix_q_lpf, name="Quadrature Filtered Line (Q)", line=dict(color="purple")), row=2, col=1)

fig.add_trace(go.Scatter(x=t, y=data_m, name="In-Phase Filtered Line (I_raw)", line=dict(color="blue")), row=7, col=1)
fig.add_trace(go.Scatter(x=t, y=data_i, name="Quadrature Filtered Line (Q_raw)", line=dict(color="purple")), row=7, col=1)

# Row 3: Physical Signal Phase Tracking
fig.add_trace(go.Scatter(x=t, y=adc_aligned, name="Raw Aligned Input (ADC)", line=dict(color="gray", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=sin_ref, name="Internal NCO Sine Wave Reference", line=dict(color="crimson", width = 1)), row=3, col=1)

# Row 4: ADPLL Edge Activity Pulses
fig.add_trace(go.Scatter(x=t_freq, y=ref_edge_pll, name="Ref PLL Input Edge Trigger", line=dict(color="orange")), row=4, col=1)
fig.add_trace(go.Scatter(x=t_freq, y=nco_edge_pll, name="NCO PLL Feedback Edge Trigger", line=dict(color="teal")), row=4, col=1)

# Row 5: Internal Calculated Period Base
fig.add_trace(go.Scatter(x=t_freq, y=period_avg, name="Calculated Average Period Frame Time", line=dict(color="chocolate")), row=5, col=1)

# Row 6: Frequency Outputs
fig.add_trace(go.Scatter(x=t_freq, y=freq_from_ftw, name="Absolute Frequency (Hz)", line=dict(color="black", width=2)), row=6, col=1)


fig.add_trace(go.Scatter(x=t_freq, y=data_i, name="adc_output", line=dict(color="black", width=2)), row=7, col=1)


# Global Layout Styling Adjustments
fig.update_layout(
    height=1500,
    title="Lock-In Amplifier & Extraction Core System Debug Performance Evaluation Dashboard",
    hovermode="x unified",
    showlegend=True
)

# Apply common axis configuration labels
for i in range(1, 8):
    fig.update_yaxes(title_text="Counts / Metrics", row=i, col=1)
fig.update_xaxes(title_text="Time (Seconds)", row=6, col=1)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [22]:
# ============================================================
# AMPLITUDE STATISTICS
# ============================================================

R = calculated_amplitude

# Basic statistics
R_mean = np.mean(R)
R_std = np.std(R)
R_min = np.min(R)
R_max = np.max(R)
R_pp = R_max - R_min          # Peak-to-peak variation


print("\n================ R STATISTICS ================")
print(f"Mean(R)               = {R_mean:.8f}")
print(f"Std(R)                = {R_std:.8f}")
print(f"Min(R)                = {R_min:.8f}")
print(f"Max(R)                = {R_max:.8f}")
print(f"Max-Min               = {R_pp:.8f}")


================ R STATISTICS ================
Mean(R)               = 0.99985033
Std(R)                = 0.00029300
Min(R)                = 0.99940437
Max(R)                = 1.00029492
Max-Min               = 0.00089055


In [23]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate

N = len(adc_aligned)

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0
FREQ_MAX_HZ = 500000.0

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal = adc_aligned * window

REMOVE_DC = True
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Main FFT trace
fig.add_trace(go.Scatter(
    x=freqs_khz,
    y=magnitude_dbv,
    mode='lines',
    name='R Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# ==========================================
# DOMINANT PEAK DETECTION
# ==========================================
from scipy.signal import find_peaks

# Find all local maxima
candidate_peaks, _ = find_peaks(magnitude_dbv)

# Sort peaks by magnitude (highest first)
candidate_peaks = candidate_peaks[
    np.argsort(magnitude_dbv[candidate_peaks])[::-1]
]

PEAK_SEPARATION_HZ = 25000.0   # Minimum separation between peaks
MAX_PEAKS = 20              # Number of peaks to display

selected_peaks = []

for p in candidate_peaks:

    peak_freq = fft_freqs[p]

    # Check whether another stronger peak already exists nearby
    keep = True

    for s in selected_peaks:
        if abs(peak_freq - fft_freqs[s]) < PEAK_SEPARATION_HZ:
            keep = False
            break

    if keep:
        selected_peaks.append(p)

    if len(selected_peaks) >= MAX_PEAKS:
        break

selected_peaks = np.array(selected_peaks)

# Sort selected peaks by frequency for cleaner display
selected_peaks = selected_peaks[
    np.argsort(freqs_khz[selected_peaks])
]

# Add markers and labels to plot
fig.add_trace(
    go.Scatter(
        x=freqs_khz[selected_peaks],
        y=magnitude_dbv[selected_peaks],
        mode='markers+text',
        text=[
            f"{freqs_khz[p]:.3f} kHz"
            for p in selected_peaks
        ],
        textposition="top center",
        marker=dict(size=10),
        name="Dominant Peaks",
        hovertemplate=
            "<b>Peak</b><br>" +
            "Freq: %{x:.6f} kHz<br>" +
            "Mag: %{y:.2f} dBV<extra></extra>"
    )
)

# Print peak table to console
print("\nDominant Peaks:")
print("-" * 40)

for p in selected_peaks:
    print(
        f"Freq = {fft_freqs[p]:12.3f} Hz    "
        f"Mag = {magnitude_dbv[p]:8.2f} dBV"
    )

# ==========================================
# Layout settings
# ==========================================
fig.update_layout(
    title="FFT Spectrum of Lock-In Vector Amplitude (R)",
    xaxis_title="Frequency (kHz)",
    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [27]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate
#calculated_amplitude_raw = calculated_amplitude_raw[:int(len(calculated_amplitude_raw)/4)]
N = len(calculated_amplitude)

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0
FREQ_MAX_HZ = 500000.0

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal =calculated_amplitude * window

REMOVE_DC = True
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Main FFT trace
fig.add_trace(go.Scatter(
    x=freqs_khz,
    y=magnitude_dbv,
    mode='lines',
    name='R Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# ==========================================
# DOMINANT PEAK DETECTION
# ==========================================
from scipy.signal import find_peaks

# Find all local maxima
candidate_peaks, _ = find_peaks(magnitude_dbv)

# Sort peaks by magnitude (highest first)
candidate_peaks = candidate_peaks[
    np.argsort(magnitude_dbv[candidate_peaks])[::-1]
]

PEAK_SEPARATION_HZ = 25000.0   # Minimum separation between peaks
MAX_PEAKS = 20              # Number of peaks to display

selected_peaks = []

for p in candidate_peaks:

    peak_freq = fft_freqs[p]

    # Check whether another stronger peak already exists nearby
    keep = True

    for s in selected_peaks:
        if abs(peak_freq - fft_freqs[s]) < PEAK_SEPARATION_HZ:
            keep = False
            break

    if keep:
        selected_peaks.append(p)

    if len(selected_peaks) >= MAX_PEAKS:
        break

selected_peaks = np.array(selected_peaks)

# Sort selected peaks by frequency for cleaner display
selected_peaks = selected_peaks[
    np.argsort(freqs_khz[selected_peaks])
]

# Add markers and labels to plot
fig.add_trace(
    go.Scatter(
        x=freqs_khz[selected_peaks],
        y=magnitude_dbv[selected_peaks],
        mode='markers+text',
        text=[
            f"{freqs_khz[p]:.3f} kHz"
            for p in selected_peaks
        ],
        textposition="top center",
        marker=dict(size=10),
        name="Dominant Peaks",
        hovertemplate=
            "<b>Peak</b><br>" +
            "Freq: %{x:.6f} kHz<br>" +
            "Mag: %{y:.2f} dBV<extra></extra>"
    )
)

# Print peak table to console
print("\nDominant Peaks:")
print("-" * 40)

for p in selected_peaks:
    print(
        f"Freq = {fft_freqs[p]:12.3f} Hz    "
        f"Mag = {magnitude_dbv[p]:8.2f} dBV"
    )

# ==========================================
# Layout settings
# ==========================================
fig.update_layout(
    title="FFT Spectrum of Lock-In Vector Amplitude (R)",
    xaxis_title="Frequency (kHz)",

    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [28]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate
N = len(data_i)//4

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0
FREQ_MAX_HZ = 500000.0

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal = data_i[0:N] * window

REMOVE_DC = True
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Main FFT trace
fig.add_trace(go.Scatter(
    x=freqs_khz,
    y=magnitude_dbv,
    mode='lines',
    name='R Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# ==========================================
# DOMINANT PEAK DETECTION
# ==========================================
from scipy.signal import find_peaks

# Find all local maxima
candidate_peaks, _ = find_peaks(magnitude_dbv)

# Sort peaks by magnitude (highest first)
candidate_peaks = candidate_peaks[
    np.argsort(magnitude_dbv[candidate_peaks])[::-1]
]

PEAK_SEPARATION_HZ = 25000.0   # Minimum separation between peaks
MAX_PEAKS = 20              # Number of peaks to display

selected_peaks = []

for p in candidate_peaks:

    peak_freq = fft_freqs[p]

    # Check whether another stronger peak already exists nearby
    keep = True

    for s in selected_peaks:
        if abs(peak_freq - fft_freqs[s]) < PEAK_SEPARATION_HZ:
            keep = False
            break

    if keep:
        selected_peaks.append(p)

    if len(selected_peaks) >= MAX_PEAKS:
        break

selected_peaks = np.array(selected_peaks)

# Sort selected peaks by frequency for cleaner display
selected_peaks = selected_peaks[
    np.argsort(freqs_khz[selected_peaks])
]

# Add markers and labels to plot
fig.add_trace(
    go.Scatter(
        x=freqs_khz[selected_peaks],
        y=magnitude_dbv[selected_peaks],
        mode='markers+text',
        text=[
            f"{freqs_khz[p]:.3f} kHz"
            for p in selected_peaks
        ],
        textposition="top center",
        marker=dict(size=10),
        name="Dominant Peaks",
        hovertemplate=
            "<b>Peak</b><br>" +
            "Freq: %{x:.6f} kHz<br>" +
            "Mag: %{y:.2f} dBV<extra></extra>"
    )
)

# Print peak table to console
print("\nDominant Peaks:")
print("-" * 40)

for p in selected_peaks:
    print(
        f"Freq = {fft_freqs[p]:12.3f} Hz    "
        f"Mag = {magnitude_dbv[p]:8.2f} dBV"
    )

# ==========================================
# Layout settings
# ==========================================
fig.update_layout(
    title="FFT Spectrum of Lock-In Vector Amplitude (R)",
    xaxis_title="Frequency (kHz)",

    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [99]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate
N = len(sin_ref)

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0
FREQ_MAX_HZ = 500000.0

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal = sin_ref * window

REMOVE_DC = True
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Main FFT trace
fig.add_trace(go.Scatter(
    x=freqs_khz,
    y=magnitude_dbv,
    mode='lines',
    name='R Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# ==========================================
# DOMINANT PEAK DETECTION
# ==========================================
from scipy.signal import find_peaks

# Find all local maxima
candidate_peaks, _ = find_peaks(magnitude_dbv)

# Sort peaks by magnitude (highest first)
candidate_peaks = candidate_peaks[
    np.argsort(magnitude_dbv[candidate_peaks])[::-1]
]

PEAK_SEPARATION_HZ = 25000.0   # Minimum separation between peaks
MAX_PEAKS = 20              # Number of peaks to display

selected_peaks = []

for p in candidate_peaks:

    peak_freq = fft_freqs[p]

    # Check whether another stronger peak already exists nearby
    keep = True

    for s in selected_peaks:
        if abs(peak_freq - fft_freqs[s]) < PEAK_SEPARATION_HZ:
            keep = False
            break

    if keep:
        selected_peaks.append(p)

    if len(selected_peaks) >= MAX_PEAKS:
        break

selected_peaks = np.array(selected_peaks)

# Sort selected peaks by frequency for cleaner display
selected_peaks = selected_peaks[
    np.argsort(freqs_khz[selected_peaks])
]

# Add markers and labels to plot
fig.add_trace(
    go.Scatter(
        x=freqs_khz[selected_peaks],
        y=magnitude_dbv[selected_peaks],
        mode='markers+text',
        text=[
            f"{freqs_khz[p]:.3f} kHz"
            for p in selected_peaks
        ],
        textposition="top center",
        marker=dict(size=10),
        name="Dominant Peaks",
        hovertemplate=
            "<b>Peak</b><br>" +
            "Freq: %{x:.6f} kHz<br>" +
            "Mag: %{y:.2f} dBV<extra></extra>"
    )
)

# Print peak table to console
print("\nDominant Peaks:")
print("-" * 40)

for p in selected_peaks:
    print(
        f"Freq = {fft_freqs[p]:12.3f} Hz    "
        f"Mag = {magnitude_dbv[p]:8.2f} dBV"
    )

# ==========================================
# Layout settings
# ==========================================
fig.update_layout(
    title="FFT Spectrum of Lock-In Vector Amplitude (R)",
    xaxis_title="Frequency (kHz)",

    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [100]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate
N = len(calculated_amplitude)

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0
FREQ_MAX_HZ = 500000.0

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal =calculated_amplitude* window

REMOVE_DC = False
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Main FFT trace
fig.add_trace(go.Scatter(
    x=freqs_khz,
    y=magnitude_dbv,
    mode='lines',
    name='R Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# ==========================================
# DOMINANT PEAK DETECTION
# ==========================================
from scipy.signal import find_peaks

# Find all local maxima
candidate_peaks, _ = find_peaks(magnitude_dbv)

# Sort peaks by magnitude (highest first)
candidate_peaks = candidate_peaks[
    np.argsort(magnitude_dbv[candidate_peaks])[::-1]
]

PEAK_SEPARATION_HZ = 25000.0   # Minimum separation between peaks
MAX_PEAKS = 20              # Number of peaks to display

selected_peaks = []

for p in candidate_peaks:

    peak_freq = fft_freqs[p]

    # Check whether another stronger peak already exists nearby
    keep = True

    for s in selected_peaks:
        if abs(peak_freq - fft_freqs[s]) < PEAK_SEPARATION_HZ:
            keep = False
            break

    if keep:
        selected_peaks.append(p)

    if len(selected_peaks) >= MAX_PEAKS:
        break

selected_peaks = np.array(selected_peaks)

# Sort selected peaks by frequency for cleaner display
selected_peaks = selected_peaks[
    np.argsort(freqs_khz[selected_peaks])
]

# Add markers and labels to plot
fig.add_trace(
    go.Scatter(
        x=freqs_khz[selected_peaks],
        y=magnitude_dbv[selected_peaks],
        mode='markers+text',
        text=[
            f"{freqs_khz[p]:.3f} kHz"
            for p in selected_peaks
        ],
        textposition="top center",
        marker=dict(size=10),
        name="Dominant Peaks",
        hovertemplate=
            "<b>Peak</b><br>" +
            "Freq: %{x:.6f} kHz<br>" +
            "Mag: %{y:.2f} dBV<extra></extra>"
    )
)

# Print peak table to console
print("\nDominant Peaks:")
print("-" * 40)

for p in selected_peaks:
    print(
        f"Freq = {fft_freqs[p]:12.3f} Hz    "
        f"Mag = {magnitude_dbv[p]:8.2f} dBV"
    )

# ==========================================
# Layout settings
# ==========================================
fig.update_layout(
    title="FFT Spectrum of Lock-In Vector Amplitude (R)",
    xaxis_title="Frequency (kHz)",
    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import numpy as np
from scipy.signal import detrend
import plotly.graph_objects as go

# ============================================================
# ===== FFT CONFIGURATION & USER LIMITS =====
# ============================================================
effective_fs = fs  # Post-decimation sample rate
N = len(mix_q_lpf)

# Define your initial zoom window here (in Hz)
FREQ_MIN_HZ = 0.0
FREQ_MAX_HZ = 500000.0

# ============================================================
# ===== FFT COMPUTATION =====
# ============================================================
window = np.hanning(N)
windowed_signal = mix_q_lpf * window

REMOVE_DC = True
if REMOVE_DC:
    fft_input = detrend(windowed_signal)
else:
    fft_input = windowed_signal

fft_raw = np.fft.rfft(fft_input)
fft_freqs = np.fft.rfftfreq(N, d=1.0/effective_fs)

magnitude_volts = (np.abs(fft_raw) / N) * 2.0 * 2.0
magnitude_dbv = 20 * np.log10(magnitude_volts + 1e-12)

# Convert arrays to kHz for easier reading on the plot
freqs_khz = fft_freqs / 1e3

# ============================================================
# ===== INTERACTIVE PLOTLY SPECTRUM =====
# ============================================================
fig = go.Figure()

# Main FFT trace
fig.add_trace(go.Scatter(
    x=freqs_khz,
    y=magnitude_dbv,
    mode='lines',
    name=' Amplitude FFT',
    line=dict(color='crimson', width=1.5),
    hovertemplate='<b>Frequency:</b> %{x:.3f} kHz<br><b>Magnitude:</b> %{y:.2f} dBV<extra></extra>'
))

# ==========================================
# DOMINANT PEAK DETECTION
# ==========================================
from scipy.signal import find_peaks

# Find all local maxima
candidate_peaks, _ = find_peaks(magnitude_dbv)

# Sort peaks by magnitude (highest first)
candidate_peaks = candidate_peaks[
    np.argsort(magnitude_dbv[candidate_peaks])[::-1]
]

PEAK_SEPARATION_HZ = 5000.0   # Minimum separation between peaks
MAX_PEAKS = 10              # Number of peaks to display

selected_peaks = []

for p in candidate_peaks:

    peak_freq = fft_freqs[p]

    # Check whether another stronger peak already exists nearby
    keep = True

    for s in selected_peaks:
        if abs(peak_freq - fft_freqs[s]) < PEAK_SEPARATION_HZ:
            keep = False
            break

    if keep:
        selected_peaks.append(p)

    if len(selected_peaks) >= MAX_PEAKS:
        break

selected_peaks = np.array(selected_peaks)

# Sort selected peaks by frequency for cleaner display
selected_peaks = selected_peaks[
    np.argsort(freqs_khz[selected_peaks])
]

# Add markers and labels to plot
fig.add_trace(
    go.Scatter(
        x=freqs_khz[selected_peaks],
        y=magnitude_dbv[selected_peaks],
        mode='markers+text',
        text=[
            f"{freqs_khz[p]:.3f} kHz"
            for p in selected_peaks
        ],
        textposition="top center",
        marker=dict(size=10),
        name="Dominant Peaks",
        hovertemplate=
            "<b>Peak</b><br>" +
            "Freq: %{x:.6f} kHz<br>" +
            "Mag: %{y:.2f} dBV<extra></extra>"
    )
)

# Print peak table to console
print("\nDominant Peaks:")
print("-" * 40)

for p in selected_peaks:
    print(
        f"Freq = {fft_freqs[p]:12.3f} Hz    "
        f"Mag = {magnitude_dbv[p]:8.2f} dBV"
    )

# ==========================================
# Layout settings
# ==========================================
fig.update_layout(
    title="FFT Spectrum of SINE REFERENCE",
    xaxis_title="Frequency (kHz)",
    xaxis=dict(range=[FREQ_MIN_HZ / 1e3, FREQ_MAX_HZ / 1e3]),
    hovermode="x unified",
    template="plotly_white"
)

fig.show()

IndexError: arrays used as indices must be of integer (or boolean) type

In [ ]:
print("Frequency resolution =", fft_freqs[1] - fft_freqs[0], "Hz")
print(fft_freqs[:10])

Frequency resolution = 125.0 Hz
[   0.  125.  250.  375.  500.  625.  750.  875. 1000. 1125.]


In [ ]:
import numpy as np
import plotly.graph_objects as go

# ============================================================
# USER SETTINGS
# ============================================================
effective_fs = fs
signal = adc_aligned

REMOVE_DC = True      # Keep DC to inspect leakage
USE_WINDOW = False      # Set False if you want to see raw leakage

# ============================================================
# PREPROCESSING
# ============================================================
if USE_WINDOW:
    signal = signal * np.hanning(len(signal))

if REMOVE_DC:
    signal = signal - np.mean(signal)

N = len(signal)

# ============================================================
# ZERO-PAD TO GET 1 Hz FFT GRID
# ============================================================
NFFT = int(effective_fs)      # Gives 1 Hz spacing

fft = np.fft.rfft(signal, n=NFFT)
freqs = np.fft.rfftfreq(NFFT, d=1/effective_fs)

# Magnitude scaling
mag = (np.abs(fft) / N) * 2
mag_db = 20 * np.log10(mag + 1e-15)

# ============================================================
# PLOT ONLY 0-1000 Hz
# ============================================================
mask = freqs <= 1000

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=freqs[mask],
        y=mag_db[mask],
        mode='lines',
        name='FFT'
    )
)

fig.update_layout(
    title="FFT Near DC (0-1 kHz)",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Magnitude (dBV)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# ============================================================
# PRINT FIRST FEW BINS
# ============================================================
print("First 20 FFT bins:")
for i in range(20):
    print(f"{freqs[i]:6.1f} Hz : {mag_db[i]:8.2f} dBV")

First 20 FFT bins:
   0.0 Hz :  -299.00 dBV
   1.0 Hz :  -127.44 dBV
   2.0 Hz :  -121.42 dBV
   3.0 Hz :  -117.89 dBV
   4.0 Hz :  -115.40 dBV
   5.0 Hz :  -113.46 dBV
   6.0 Hz :  -111.88 dBV
   7.0 Hz :  -110.54 dBV
   8.0 Hz :  -109.38 dBV
   9.0 Hz :  -108.36 dBV
  10.0 Hz :  -107.45 dBV
  11.0 Hz :  -106.63 dBV
  12.0 Hz :  -105.87 dBV
  13.0 Hz :  -105.18 dBV
  14.0 Hz :  -104.54 dBV
  15.0 Hz :  -103.95 dBV
  16.0 Hz :  -103.39 dBV
  17.0 Hz :  -102.87 dBV
  18.0 Hz :  -102.38 dBV
  19.0 Hz :  -101.92 dBV


In [ ]:
for f in [0, 125, 250, 375, 500]:
    idx = np.argmin(np.abs(freqs - f))
    print(f"{freqs[idx]:6.1f} Hz : {mag_db[idx]:8.2f} dBV")

   0.0 Hz :   -14.31 dBV
 125.0 Hz :   -87.86 dBV
 250.0 Hz :   -91.93 dBV
 375.0 Hz :   -95.73 dBV
 500.0 Hz :  -109.71 dBV


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.signal import detrend

# ============================================================
# USER SETTINGS
# ============================================================

signal = adc_aligned
fs = effective_fs

REMOVE_DC = False

WINDOW = "hann"      # Options: "none", "hann", "hamming", "blackman"

ZERO_PAD_FACTOR = 1  # 1 = no zero-padding
                     # 4 = 4× interpolation
                     # 8 = 8× interpolation

PLOT_MIN = 0
PLOT_MAX = fs / 200

# ============================================================
# PREPROCESS
# ============================================================

signal = np.asarray(signal, dtype=float)

if REMOVE_DC:
    signal = detrend(signal, type='constant')

N = len(signal)

# ------------------------------------------------------------
# Window Selection
# ------------------------------------------------------------

if WINDOW.lower() == "hann":
    window = np.hanning(N)
elif WINDOW.lower() == "hamming":
    window = np.hamming(N)
elif WINDOW.lower() == "blackman":
    window = np.blackman(N)
else:
    window = np.ones(N)

coherent_gain = np.mean(window)

signal_windowed = signal * window

# ============================================================
# FFT
# ============================================================

NFFT = int(N * ZERO_PAD_FACTOR)

fft = np.fft.rfft(signal_windowed, n=NFFT)

freq = np.fft.rfftfreq(NFFT, d=1/fs)

# ============================================================
# Amplitude Scaling
# ============================================================

mag = np.abs(fft)

mag = mag / (N * coherent_gain)

# Double all bins except DC and Nyquist
if len(mag) > 2:
    mag[1:-1] *= 2

mag_db = 20 * np.log10(mag + 1e-15)

# ============================================================
# Plot
# ============================================================

mask = (freq >= PLOT_MIN) & (freq <= PLOT_MAX)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=freq[mask],
        y=mag_db[mask],
        mode='lines',
        name='FFT',
        line=dict(width=1.5)
    )
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
    title="FFT Spectrum",
    xaxis_title="Frequency (Hz)",
    yaxis_title="Magnitude (dBV)"
)

fig.show()

# ============================================================
# FFT Information
# ============================================================

print("\n========== FFT INFORMATION ==========")
print(f"Samples                : {N}")
print(f"Sampling Frequency     : {fs:,.2f} Hz")
print(f"FFT Size               : {NFFT}")
print(f"True Resolution        : {fs/N:.6f} Hz")
print(f"Displayed Grid Spacing : {fs/NFFT:.6f} Hz")
print(f"Window                 : {WINDOW}")
print(f"Coherent Gain          : {coherent_gain:.6f}")


========== FFT INFORMATION ==========
Samples                : 1000000
Sampling Frequency     : 125,000,000.00 Hz
FFT Size               : 1000000
True Resolution        : 125.000000 Hz
Displayed Grid Spacing : 125.000000 Hz
Window                 : hann
Coherent Gain          : 0.500000
